<!--
Derived from garipovroma/autojudge (Apache-2.0):
  https://github.com/garipovroma/autojudge/blob/cf180e89a8718e10a34ccf222a2a96e786e24a20/notebooks/train.ipynb
Modifications for SelfJudge:
  - loads and merges several mined datasets (GSM8K + LiveCodeBench) and trains on
    target-model hidden states (target_hiddens) with AutoJudge's answer-match labels.
  - added ROC / recall-constrained threshold selection and checkpoint naming for
    verifier_general/*-auto-logistic.pkl.
-->

In [2]:
from typing import Union, Sequence, Tuple, Dict, Literal
import json
import os

from tqdm.auto import tqdm
from termcolor import colored
import numpy as np
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F

import transformers

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

import pickle

class args:
    random_seed = 52
    train_size = 0.9
    
    # Llama3
    # data_path_gsm = f'compare/important_tokens--model-selfjudge_nextfew20-500_samples_0_with_hidden-new.pt' 
    # data_path_lcb = f'compare_lcb/important_tokens--model-selfjudge_nextfew_all-220_samples_0_with_hidden_new.pt'

    # QWEN
    data_path_gsm = f'important_tokens/gsm8k-500_important_tokens-qwen2.5-self_judge.pt' 
    data_path_lcb = f'important_tokens/lcb-220_important_tokens-qwen2.5-self_judge.pt'
    

In [3]:
data_path_list = [args.data_path_gsm, args.data_path_lcb]

for i, data_path in enumerate(data_path_list):
    if i == 0:
        data_list = torch.load(data_path)
    else:
        data_list = data_list + torch.load(data_path)


/tmp/ipykernel_140918/2729919774.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data_list = torch.load(data_path)
/tmp/ipykernel_140918/2729919774.py:7: FutureWarning: 

In [4]:
np.random.seed(args.random_seed)

In [5]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:75% !important; }</style>"))

In [6]:
# def make_X_y(data: list[Dict], hiddens: Literal['current', 'prev', 'both'] = 'current') -> Tuple[np.array, np.array]: 
def make_X_y(data, hiddens='current'): 
    X = []
    X_prev = []
    y = []

    n_skip = 0
    skip_indexes = set()
    for sample_idx, sample_dict in enumerate(data):
        # if 'hiddens' in sample_dict:
        if 'target_hiddens' in sample_dict:
            if len(sample_dict['changed_token_indices']) != len(sample_dict['target_hiddens']):
                continue
            # x_ = [i.numpy() for i in sample_dict['hiddens']]
            x_ = [i.numpy() for i in sample_dict['target_hiddens']]
            x_prev_ = [i.numpy() for i in sample_dict['target_prev_hiddens']]
            y_ = [int(i[1]) for i in sample_dict['changed_token_indices']]
            X.extend(x_)
            X_prev.extend(x_prev_)
            y.extend(y_)
        else:
            n_skip += 1
            skip_indexes.add(sample_idx)


    print(skip_indexes)
        
    X = np.array(X)
    X_prev = np.array(X_prev)
    y = np.array(y)
    if hiddens == 'current':
        return X, y
    elif hiddens == 'prev':
        return X_prev, y
    elif hiddens == 'both':
        X = np.concatenate((X, X_prev), axis=1)
        return X, y
    else:
        raise ValueError(f'{hiddens=} is not supported')

## Load and create data

In [7]:
data = data_list
np.random.shuffle(data)
first_test_sample_idx = int(len(data) * args.train_size) + 1

train_data = data[:first_test_sample_idx]
test_data = data[first_test_sample_idx:]

In [8]:
len(train_data), len(test_data)

(649, 71)

In [9]:
train_data[0].keys()

dict_keys(['changed_token_indices', 'colored_tokens', 'draft_score', 'target_score', 'current_response', 'likelihood_diffs_0', 'likelihood_diffs_5', 'likelihood_diffs_10', 'likelihood_diffs_20', 'global_id', 'question_id', 'contest_id', 'contest_date', 'prompt', 'tags', 'task', 'source', 'beam_size', 'id', 'target_hiddens', 'target_prev_hiddens', 'target_hiddens_replaced', 'target_prev_hiddens_replaced', 'prompt_hidden'])

In [10]:
X_train, y_train = make_X_y(train_data)
X_val, y_val = make_X_y(test_data)

set()
set()


In [11]:
len(X_train), len(X_val)

(22588, 2160)

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

(22588, 3584) (2160, 3584)
0.4734814945989021 0.45601851851851855


## train

In [13]:
# def train_head_and_search_best_hparam(hidden_dim_slice: slice, C: float | None = None):
def train_head_and_search_best_hparam(hidden_dim_slice, C=None):
    C_grid = [C] if C is not None else (10**0.5) ** -np.arange(10)[::-1]

    dataframes = []

    best_C = None
    best_metric_roc_auc = -1
    best_dataframe = None

    best_model = None
    for c_idx, C in enumerate(tqdm(C_grid)):

        print('#####' * 6, end='')
        print(f' {C=:.5f}, {c_idx=} ', end='')
        print('#####' * 6)    

        model_ = LogisticRegression(C=C)

        model_.fit(X_train[:, hidden_dim_slice], y_train)
        train_probs = model_.predict_proba(X_train[:, hidden_dim_slice])[:, 1]
        val_probs = model_.predict_proba(X_val[:, hidden_dim_slice])[:, 1]
        quantiles =  np.linspace(0, 1, 20)
        # quantiles =  np.linspace(0, 1, 1000)
        roc_auc_score_train = 0.0 if len(set(y_val)) == 1 else roc_auc_score(y_train, train_probs)
        roc_auc_score_val = 0.0 if len(set(y_val)) == 1 else roc_auc_score(y_val, val_probs)
        metrics = {
            'roc_auc_train': [],
            'roc_auc_val': [],

            'acc_train': [],
            'recall_train': [],
            'precision_train': [],

            'acc_val': [],
            'recall_val': [],
            'precision_val': [],

            'neg_rate_val': [],

            'q': [],
            'thr': []
        }
        quantiles = list(quantiles)
        quantiles = sorted(quantiles)
        for quantile in quantiles:
            thr = np.quantile(val_probs, quantile)
            
            train_pred = train_probs > thr
            val_pred = val_probs > thr

            metrics['acc_train'].append(accuracy_score(y_train, train_pred))
            metrics['recall_train'].append(recall_score(y_train, train_pred))
            metrics['precision_train'].append(precision_score(y_train, train_pred))

            metrics['acc_val'].append(accuracy_score(y_val, val_pred))
            metrics['recall_val'].append(recall_score(y_val, val_pred))
            metrics['precision_val'].append(precision_score(y_val, val_pred))

            metrics['roc_auc_train'].append(roc_auc_score_train)
            metrics['roc_auc_val'].append(roc_auc_score_val)

            metrics['q'].append(quantile)
            metrics['thr'].append(thr)

            metrics['neg_rate_val'].append(1 - (val_probs > thr).mean())

        metrics_df = pd.DataFrame(metrics).set_index('q')
        metrics_df['C'] = C
        metrics_df['c_idx'] = c_idx
    
        print(metrics_df[['roc_auc_train', 'roc_auc_val', 'C']].iloc[0])
        print()
        if roc_auc_score_val > best_metric_roc_auc:
            best_metric_roc_auc = roc_auc_score_val
            best_C = C
            best_dataframe = metrics_df
            best_model = model_

        dataframes.append(metrics_df)

    dataframes = pd.concat(dataframes)

    print(f"{best_C=} {best_metric_roc_auc=}")

    return dataframes, best_dataframe, best_model


In [14]:
all_dataframes_1_token, best_datarame_1_token, best_head_1_token = train_head_and_search_best_hparam(hidden_dim_slice=slice(2048*3))

  0%|          | 0/10 [00:00<?, ?it/s]

############################## C=0.00003, c_idx=0 ##############################


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


roc_auc_train    0.865761
roc_auc_val      0.775480
C                0.000032
Name: 0.0, dtype: float64

############################## C=0.00010, c_idx=1 ##############################


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

/usr/local/lib/python3.8/dist-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


roc_auc_train    0.947473
roc_auc_val      0.724669
C                0.316228
Name: 0.0, dtype: float64

############################## C=1.00000, c_idx=9 ##############################


/usr/local/lib/python3.8/dist-packages/sklearn/linear_model/_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


roc_auc_train    0.947502
roc_auc_val      0.723369
C                1.000000
Name: 0.0, dtype: float64

best_C=9.999999999999995e-05 best_metric_roc_auc=0.7821950534614968


/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [15]:
from sklearn.metrics import (
    roc_curve, precision_recall_curve, f1_score, 
    precision_score, recall_score, roc_auc_score, confusion_matrix
)
import numpy as np

In [16]:
y_probs = best_head_1_token.predict_proba(X_val)[:, 1]

fpr, tpr, thresholds = roc_curve(y_val, y_probs)
youden_j = tpr - fpr
optimal_idx = np.argmax(youden_j)
youden_threshold = thresholds[optimal_idx]

print(f"Optimal threshold (Youden): {youden_threshold:.3f}")
print(f"TPR: {tpr[optimal_idx]:.3f}, FPR: {fpr[optimal_idx]:.3f}")


Optimal threshold (Youden): 0.411
TPR: 0.909, FPR: 0.464


In [17]:
thresholds = np.arange(0.001, 1.0, 0.001)
min_recall=0.8

valid_thresholds = []

for threshold in thresholds:
    y_pred = (y_probs >= threshold).astype(int)
    recall = recall_score(y_val, y_pred, zero_division=0)
    
    if recall >= min_recall:
        precision = precision_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        valid_thresholds.append((threshold, recall, precision, f1, fn, fp, tn, tp))
    
best_threshold_data = max(valid_thresholds, key=lambda x: x[2])
min_recall_threshold = best_threshold_data[0]

print(f"Optimal threshold (recall): {min_recall_threshold:.3f}")

Optimal threshold (recall): 0.522


### Save checkpoint

In [18]:
# checkpoint_path_1_token = os.path.join(f'../verifier_general', f'Llama-3.1-8B-auto-logistic.pkl')
checkpoint_path_1_token = os.path.join(f'../verifier_general', f'qwen2.5-7b-auto-logistic.pkl')

# save
with open(checkpoint_path_1_token, 'wb') as f: # <- We use this model in our main experiments
    dump_dict = dict(model=best_head_1_token, scaler=scaler, youden_threshold=youden_threshold, min_recall_threshold=min_recall_threshold)
    pickle.dump(dump_dict, f)

